In [ ]:
### Evaluation and Metrics Module

In [ ]:
# imports
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

In [ ]:
class Evaluator:
    def __init__(self, model, device=None):
        self.model = model
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def evaluate_accuracy(self, dataloader):
        self.model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for data, target in dataloader:
                data = data.to(self.device)
                target = target.to(self.device)

                output = self.model(data)
                preds = output.argmax(dim=1)

                correct += (preds == target).sum().item()
                total += target.size(0)

        return correct / total

    def get_predictions(self, dataloader):
        self.model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for data, target in dataloader:
                data = data.to(self.device)
                target = target.to(self.device)

                output = self.model(data)
                preds = output.argmax(dim=1)

                all_preds.append(preds.cpu())
                all_labels.append(target.cpu())

        return torch.cat(all_preds), torch.cat(all_labels)

    def confusion_matrix(self, dataloader, num_classes=10):
        preds, labels = self.get_predictions(dataloader)

        cm = torch.zeros((num_classes, num_classes), dtype=torch.int32)

        for t, p in zip(labels, preds):
            cm[t, p] += 1

        return cm